# Task 5: Model training under leave-one-province-out CV
Rules enforced in every fold:
- Preprocessing is re-fit on fold-train rows only (nested, no leakage)
- Augmentation is generated from fold-train rows only
- Held-out province rows are real, untouched, and used only for scoring
- All 4 models see the identical synthetic rows within a fold (fair comparison)
Models: Linear (reported baseline), Ridge, Random Forest, constrained LightGBM.

In [1]:
# setup, data, and model roaster
import sys, json, time
from pathlib import Path
import numpy as np, pandas as pd, joblib

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.utils.augmentation import stratified_mixup

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv(PROJECT_ROOT / "data/processed/merged_rice_yield_dataset.csv")
df["temp_mean_c"] = (df["temp_max_c"] + df["temp_min_c"]) / 2
df["temp_range_c"] = df["temp_max_c"] - df["temp_min_c"]

split_info = json.load(open(PROJECT_ROOT / "artifacts/split_provinces.json"))
df_train = df[df["province"].isin(split_info["train_provinces"])].copy()

X_train_raw = df_train.drop(columns=["yield_mt_ha", "province"])
y_train = df_train["yield_mt_ha"].to_numpy()
strata = (df_train["season_period"] + "|" + df_train["ecosystem"]
          + "|" + df_train["island_group"]).to_numpy()
province_of_row = df_train["province"].to_numpy()
provinces = sorted(df_train["province"].unique())

CAT_COLS = ["season_period", "ecosystem", "island_group"]
NUM_STD = ["n_application_rate", "p_application_rate", "k_application_rate",
           "total_fertilizer_kg_ha", "fertilizer_diversification_index",
           "rainfall_mm", "solar_radiation_mj_m2", "soil_ph",
           "temp_mean_c", "temp_range_c"]
NUM_LOG = ["area_harvested_ha"]

def make_preprocessor():
    return ColumnTransformer(transformers=[
        ("num_std", Pipeline([("scaler", StandardScaler())]), NUM_STD),
        ("num_log", Pipeline([("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                              ("scaler", StandardScaler())]), NUM_LOG),
        ("cat", Pipeline([("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))]), CAT_COLS),
    ])

def make_models():
    return {
        "linear_baseline": LinearRegression(),
        "ridge": Ridge(alpha=1.0, random_state=42),
        "random_forest": RandomForestRegressor(
            n_estimators=500, min_samples_leaf=2, random_state=42, n_jobs=-1),
        "lightgbm": LGBMRegressor(
            n_estimators=200, learning_rate=0.05, num_leaves=7, max_depth=3,
            min_child_samples=25, colsample_bytree=0.8,
            reg_alpha=1.0, reg_lambda=5.0, random_state=42, verbose=-1),
    }

print(f"Training provinces: {len(provinces)} | rows: {len(df_train)}")

Training provinces: 60 | rows: 229


In [2]:
# LOPO cv engine
def lopo_cv(multiplier, province_list, base_seed=42):
    model_names = list(make_models().keys())
    oof = {name: np.full(len(df_train), np.nan) for name in model_names}
    fold_rmse = {name: [] for name in model_names}
    fold_mae = {name: [] for name in model_names}

    for f, prov in enumerate(province_list):
        tr = province_of_row != prov
        va = province_of_row == prov

        prep = make_preprocessor()
        Xf = prep.fit_transform(X_train_raw[tr])
        Xv = prep.transform(X_train_raw[va])
        if hasattr(Xf, "toarray"):
            Xf, Xv = Xf.toarray(), Xv.toarray()
        yf, yv, sf = y_train[tr], y_train[va], strata[tr]

        Xa, ya, wa = stratified_mixup(Xf, yf, sf, multiplier=multiplier,
                                      random_state=base_seed + f)

        for name, model in make_models().items():
            model.fit(Xa, ya, sample_weight=wa)
            pred = model.predict(Xv)
            oof[name][va] = pred
            fold_rmse[name].append(np.sqrt(mean_squared_error(yv, pred)))
            fold_mae[name].append(mean_absolute_error(yv, pred))

    return oof, fold_rmse, fold_mae

In [3]:
# metrics summary
def summarize(oof, fold_rmse, fold_mae, multiplier):
    rows = []
    for name in oof:
        mask = ~np.isnan(oof[name])
        yt, yp = y_train[mask], oof[name][mask]
        rows.append({
            "model": name, "multiplier": multiplier,
            "oof_rmse": np.sqrt(mean_squared_error(yt, yp)),
            "oof_mae": mean_absolute_error(yt, yp),
            "oof_r2": r2_score(yt, yp),
            "fold_rmse_mean": np.mean(fold_rmse[name]),
            "fold_rmse_std": np.std(fold_rmse[name]),
            "fold_mae_mean": np.mean(fold_mae[name]),
            "fold_mae_std": np.std(fold_mae[name]),
            "n_folds": len(fold_rmse[name]),
        })
    return pd.DataFrame(rows)

In [4]:
# full sweep configuration
import time

SMOKE = False
mults = [1, 2, 3, 5]
prov_list = provinces

results, oof_store = [], {}
t0 = time.time()

for m in mults:
    print(f"Running multiplier {m}x...")
    oof, frmse, fmae = lopo_cv(m, prov_list)
    results.append(summarize(oof, frmse, fmae, m))
    
    for name in oof:
        oof_store[f"{name}_m{m}"] = oof[name]
        
print(f"\nTotal Elapsed: {time.time() - t0:.1f}s")

metrics_df = pd.concat(results)
print("\n--- Full LOPO CV Results ---")
print(metrics_df.round(4).to_string(index=False))

metrics_df.to_csv(PROJECT_ROOT / "artifacts/model_metrics.csv", index=False)

oof_df = pd.DataFrame(oof_store)
oof_df.to_csv(PROJECT_ROOT / "artifacts/oof_predictions.csv", index=False)

print("\nSaved: artifacts/model_metrics.csv")
print("Saved: artifacts/oof_predictions.csv")

Running multiplier 1x...
Running multiplier 2x...
Running multiplier 3x...
Running multiplier 5x...

Total Elapsed: 148.4s

--- Full LOPO CV Results ---
          model  multiplier  oof_rmse  oof_mae  oof_r2  fold_rmse_mean  fold_rmse_std  fold_mae_mean  fold_mae_std  n_folds
linear_baseline           1    0.6772   0.5511  0.3673          0.6115         0.2930         0.5527        0.2820       60
          ridge           1    0.6749   0.5488  0.3716          0.6094         0.2920         0.5503        0.2814       60
  random_forest           1    0.6758   0.5334  0.3700          0.6068         0.3006         0.5351        0.2887       60
       lightgbm           1    0.7027   0.5589  0.3188          0.6319         0.3150         0.5622        0.3005       60
linear_baseline           2    0.6816   0.5552  0.3591          0.6159         0.2926         0.5561        0.2831       60
          ridge           2    0.6798   0.5533  0.3625          0.6140         0.2921         0.5542   

## Copula-based Augmentation Experiment

In [5]:
import sdv
from pathlib import Path
import sys

print("sdv version:", sdv.__version__)

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.utils.augmentation_copula import copula_augment

RAW_FEAT_COLS = CAT_COLS + NUM_STD + NUM_LOG 

toy = df_train[RAW_FEAT_COLS + ["yield_mt_ha"]].head(40).copy()
toy_synth, toy_w = copula_augment(toy, multiplier=2, random_state=0)

print("Toy synthetic shape:", toy_synth.shape)
print(toy_synth.head(3).round(2))
print("PROBE OK")

sdv version: 1.38.5
Toy synthetic shape: (40, 15)
     season_period      ecosystem island_group  n_application_rate  \
0  2021_Semester_2  Non-irrigated        Luzon               68.88   
1  2021_Semester_2  Non-irrigated        Luzon              103.66   
2  2021_Semester_2      Irrigated        Luzon               66.35   

   p_application_rate  k_application_rate  total_fertilizer_kg_ha  \
0               16.73               10.39                   65.89   
1               18.31               14.09                  345.53   
2               18.28               18.88                  242.92   

   fertilizer_diversification_index  rainfall_mm  solar_radiation_mj_m2  \
0                              1.08       1931.8                  16.17   
1                              1.14       1759.3                  16.01   
2                              0.99       1858.5                  16.46   

   soil_ph  temp_mean_c  temp_range_c  area_harvested_ha  yield_mt_ha  
0     5.49        2

c:\Users\Charles\Documents\rice-yield-prediction\.venv\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


In [6]:
full_raw = df_train[RAW_FEAT_COLS + ["yield_mt_ha"]].copy()
syn3, w3 = copula_augment(full_raw, multiplier=3, random_state=42)

strata_real = (df_train["season_period"] + "|" + df_train["ecosystem"] + "|"
               + df_train["island_group"]).value_counts(normalize=True).round(3)
strata_syn = (syn3["season_period"] + "|" + syn3["ecosystem"] + "|"
              + syn3["island_group"]).value_counts(normalize=True).round(3)
print(pd.DataFrame({"real_share": strata_real, "synth_share": strata_syn}).fillna(0))

for a, b in [("total_fertilizer_kg_ha", "yield_mt_ha"),
             ("rainfall_mm", "yield_mt_ha"),
             ("soil_ph", "yield_mt_ha")]:
    print(f"corr({a}, yield): real {full_raw[[a, b]].corr().iloc[0, 1]:.3f} | "
          f"synth {syn3[[a, b]].corr().iloc[0, 1]:.3f}")

print("Synthetic yield range:", round(syn3["yield_mt_ha"].min(), 2),
      round(syn3["yield_mt_ha"].max(), 2))

c:\Users\Charles\Documents\rice-yield-prediction\.venv\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


                                        real_share  synth_share
2021_Semester_2|Irrigated|Luzon              0.127        0.131
2021_Semester_2|Irrigated|Mindanao           0.079        0.111
2021_Semester_2|Irrigated|Visayas            0.057        0.035
2021_Semester_2|Non-irrigated|Luzon          0.122        0.122
2021_Semester_2|Non-irrigated|Mindanao       0.079        0.085
2021_Semester_2|Non-irrigated|Visayas        0.052        0.035
2022_Semester_1|Irrigated|Luzon              0.127        0.109
2022_Semester_1|Irrigated|Mindanao           0.079        0.094
2022_Semester_1|Irrigated|Visayas            0.057        0.037
2022_Semester_1|Non-irrigated|Luzon          0.092        0.098
2022_Semester_1|Non-irrigated|Mindanao       0.079        0.109
2022_Semester_1|Non-irrigated|Visayas        0.052        0.033
corr(total_fertilizer_kg_ha, yield): real 0.347 | synth 0.376
corr(rainfall_mm, yield): real -0.117 | synth -0.123
corr(soil_ph, yield): real 0.412 | synth 0.393
Synthe

In [14]:
def lopo_cv_copula(multiplier, province_list, base_seed=42):
    model_names = list(make_models().keys())
    oof = {name: np.full(len(df_train), np.nan) for name in model_names}
    fold_rmse = {name: [] for name in model_names}
    fold_mae = {name: [] for name in model_names}
    raw_all = df_train[RAW_FEAT_COLS + ["yield_mt_ha"]]

    for f, prov in enumerate(province_list):
        tr, va = province_of_row != prov, province_of_row == prov

        df_s, w_s = copula_augment(raw_all[tr], multiplier=multiplier,
                                   random_state=base_seed + f)

        prep = make_preprocessor()
        Xf = prep.fit_transform(df_train[tr].drop(columns=["yield_mt_ha", "province"]))
        Xv = prep.transform(df_train[va].drop(columns=["yield_mt_ha", "province"]))
        Xs = prep.transform(df_s[RAW_FEAT_COLS]) if len(df_s) else np.empty((0, Xf.shape[1]))
        if hasattr(Xf, "toarray"):
            Xf, Xv = Xf.toarray(), Xv.toarray()
            Xs = Xs.toarray() if hasattr(Xs, "toarray") else Xs

        yf, yv = y_train[tr], y_train[va]
        Xa = np.vstack([Xf, Xs])
        ya = np.concatenate([yf, df_s["yield_mt_ha"].to_numpy()]) if len(df_s) else yf
        wa = np.concatenate([np.ones(int(tr.sum())), w_s])

        for name, model in make_models().items():
            model.fit(Xa, ya, sample_weight=wa)
            pred = model.predict(Xv)
            oof[name][va] = pred
            fold_rmse[name].append(np.sqrt(mean_squared_error(yv, pred)))
            fold_mae[name].append(mean_absolute_error(yv, pred))

    return oof, fold_rmse, fold_mae

In [15]:
if "oof_store" not in globals():   # survive kernel restarts
    _old = pd.read_csv(PROJECT_ROOT / "artifacts/oof_predictions.csv")
    oof_store = {c: _old[c].to_numpy() for c in _old.columns}

SMOKE_C = False
cop_mults = [3] if SMOKE_C else [2, 3, 5]
cop_provs = provinces[:5] if SMOKE_C else provinces

cop_results = []
t0 = time.time()
for m in cop_mults:
    print(f"Copula multiplier {m}x ...")
    oof, frmse, fmae = lopo_cv_copula(m, cop_provs)
    dfm = summarize(oof, frmse, fmae, m)
    dfm["technique"] = "copula"
    cop_results.append(dfm)
    if not SMOKE_C:
        for name in oof:
            oof_store[f"{name}_copula_m{m}"] = oof[name]
print(f"Elapsed: {time.time() - t0:.1f}s")
print(pd.concat(cop_results).round(4).to_string(index=False))

Copula multiplier 2x ...


c:\Users\Charles\Documents\rice-yield-prediction\.venv\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
c:\Users\Charles\Documents\rice-yield-prediction\.venv\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
c:\Users\Charles\Documents\rice-yield-prediction\.venv\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
c:\Users\Charles\Documents\rice-yield-prediction\.venv\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
c:\Users\Charles\Documents\rice-yield-prediction

Copula multiplier 3x ...


c:\Users\Charles\Documents\rice-yield-prediction\.venv\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
c:\Users\Charles\Documents\rice-yield-prediction\.venv\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
c:\Users\Charles\Documents\rice-yield-prediction\.venv\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
c:\Users\Charles\Documents\rice-yield-prediction\.venv\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
c:\Users\Charles\Documents\rice-yield-prediction

Copula multiplier 5x ...


c:\Users\Charles\Documents\rice-yield-prediction\.venv\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
c:\Users\Charles\Documents\rice-yield-prediction\.venv\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
c:\Users\Charles\Documents\rice-yield-prediction\.venv\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
c:\Users\Charles\Documents\rice-yield-prediction\.venv\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
c:\Users\Charles\Documents\rice-yield-prediction

Elapsed: 162.3s
          model  multiplier  oof_rmse  oof_mae  oof_r2  fold_rmse_mean  fold_rmse_std  fold_mae_mean  fold_mae_std  n_folds technique
linear_baseline           2    0.6771   0.5443  0.3674          0.6132         0.2929         0.5474        0.2853       60    copula
          ridge           2    0.6770   0.5436  0.3676          0.6130         0.2931         0.5467        0.2858       60    copula
  random_forest           2    0.7091   0.5586  0.3063          0.6324         0.3220         0.5592        0.3008       60    copula
       lightgbm           2    0.7109   0.5598  0.3027          0.6303         0.3261         0.5597        0.3085       60    copula
linear_baseline           3    0.6844   0.5479  0.3538          0.6207         0.2950         0.5516        0.2896       60    copula
          ridge           3    0.6844   0.5475  0.3539          0.6207         0.2950         0.5513        0.2897       60    copula
  random_forest           3    0.7023   0.5527

In [18]:
oof_df = pd.read_csv(PROJECT_ROOT / "artifacts/oof_predictions.csv")

rows = []
for col in oof_df.columns:
    model_part, mult = col.rsplit("_m", 1)
    mult = int(mult)
    if model_part.endswith("_copula"):
        technique, model = "copula", model_part[: -len("_copula")]
    elif mult == 1:
        technique, model = "none", model_part
    else:
        technique, model = "mixup", model_part

    yp = oof_df[col].to_numpy()
    fold_rmse, fold_mae = [], []
    for p in provinces:
        m = province_of_row == p
        fold_rmse.append(np.sqrt(mean_squared_error(y_train[m], yp[m])))
        fold_mae.append(mean_absolute_error(y_train[m], yp[m]))

    rows.append({
        "technique": technique, "model": model, "multiplier": mult,
        "oof_rmse": np.sqrt(mean_squared_error(y_train, yp)),
        "oof_mae": mean_absolute_error(y_train, yp),
        "oof_r2": r2_score(y_train, yp),
        "fold_rmse_mean": np.mean(fold_rmse), "fold_rmse_std": np.std(fold_rmse),
        "fold_mae_mean": np.mean(fold_mae), "fold_mae_std": np.std(fold_mae),
        "n_folds": len(fold_rmse),
    })

col_order = ["technique", "model", "multiplier", "oof_rmse", "oof_mae", "oof_r2",
             "fold_rmse_mean", "fold_rmse_std", "fold_mae_mean", "fold_mae_std", "n_folds"]
metrics_clean = (pd.DataFrame(rows)[col_order]
                 .sort_values(["technique", "multiplier", "model"]))
metrics_clean.to_csv(PROJECT_ROOT / "artifacts/model_metrics.csv", index=False)

print(f"Clean table rows: {len(metrics_clean)} (expect 28)")
print("\nFINAL selection view:")
print(metrics_clean.groupby(["technique", "multiplier"])["oof_rmse"].mean().round(4))

Clean table rows: 28 (expect 28)

FINAL selection view:
technique  multiplier
copula     2             0.6936
           3             0.6960
           5             0.7010
mixup      2             0.6873
           3             0.6775
           5             0.6905
none       1             0.6826
Name: oof_rmse, dtype: float64


In [19]:
import numpy as np

oof_df = pd.read_csv(PROJECT_ROOT / "artifacts/oof_predictions.csv")
y_true = y_train 

configs = {
    "Ridge + Copula 3x": "ridge_copula_m3",
    "Ridge + Mixup 3x": "ridge_m3",
    "Ridge + None 1x": "ridge_m1",
    "RF + Copula 3x": "random_forest_copula_m3",
    "RF + Mixup 3x": "random_forest_m3"
}

def get_rmse_ci(y_true, y_pred, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    boot_rmses = []
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        rmse = np.sqrt(np.mean((y_true[idx] - y_pred[idx])**2))
        boot_rmses.append(rmse)
    return np.percentile(boot_rmses, 2.5), np.percentile(boot_rmses, 97.5)

results = []
for label, col in configs.items():
    y_pred = oof_df[col].to_numpy()
    rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    lower, upper = get_rmse_ci(y_true, y_pred)
    results.append({
        "Setting": label, 
        "RMSE": round(rmse, 4), 
        "CI_2.5%": round(lower, 4), 
        "CI_97.5%": round(upper, 4)
    })

ci_df = pd.DataFrame(results)
print("--- Bootstrap 95% Confidence Intervals ---")
print(ci_df.to_string(index=False))
ci_df.to_csv(PROJECT_ROOT / "artifacts/bootstrap_cis.csv", index=False)
print("\nSaved: artifacts/bootstrap_cis.csv")

--- Bootstrap 95% Confidence Intervals ---
          Setting   RMSE  CI_2.5%  CI_97.5%
Ridge + Copula 3x 0.6844   0.6231    0.7377
 Ridge + Mixup 3x 0.6669   0.6079    0.7184
  Ridge + None 1x 0.6749   0.6150    0.7280
   RF + Copula 3x 0.7023   0.6349    0.7606
    RF + Mixup 3x 0.6838   0.6188    0.7437

Saved: artifacts/bootstrap_cis.csv


In [21]:
import joblib

# 1. Fit the preprocessor on the ENTIRE training set (all 229 rows)
final_prep = make_preprocessor()
X_all_proc = final_prep.fit_transform(X_train_raw)
if hasattr(X_all_proc, "toarray"):
    X_all_proc = X_all_proc.toarray()

# 2. Generate the winning Mixup 3x synthetic rows on the processed data
# (stratified_mixup handles the weights automatically: real=1.0, synth=0.5)
X_final, y_final, w_final = stratified_mixup(
    X_all_proc, y_train, strata, multiplier=3, random_state=42)

print(f"Final training set: {len(y_final)} rows "
      f"({len(y_train)} real + {len(y_final) - len(y_train)} synthetic)")

# 3. Train and save the 4 final models
for name, model in make_models().items():
    model.fit(X_final, y_final, sample_weight=w_final)
    joblib.dump(model, PROJECT_ROOT / f"models/model_{name}.joblib")
    print(f"Saved: models/model_{name}.joblib")

# 4. Save the final preprocessing pipeline
joblib.dump(final_prep, PROJECT_ROOT / "models/preprocessing_pipeline.joblib")
print("Saved: models/preprocessing_pipeline.joblib (fitted on all 229 train rows)")

Final training set: 687 rows (229 real + 458 synthetic)
Saved: models/model_linear_baseline.joblib
Saved: models/model_ridge.joblib
Saved: models/model_random_forest.joblib
Saved: models/model_lightgbm.joblib
Saved: models/preprocessing_pipeline.joblib (fitted on all 229 train rows)
